## Creating Association Rules from a Retail Store

### Installing and importing libraries 

![MLxtend](Mlxtend.png)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

## Step 1. Load and prepare the store dataset

### E.1 Load store_ready csv file and assign it to  variable 'store'

In [2]:
store = pd.read_csv('store_ready.csv')

**Shape**

In [3]:
store.shape

(7501, 121)

**First 2 rows**

In [4]:
store.head(2)

,Unnamed: 0,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,False,True,True,False,True,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


**First 5 Column names**

In [5]:
list(store.columns)[0:5]

['Unnamed: 0', ' asparagus', 'almonds', 'antioxydant juice', 'asparagus']

### E2. Data Preperation

**Drop 'Unnamed: 0' and also ' asparagus' columns (there is a duplicated 'asparagus' column)**

In [6]:
store.drop(['Unnamed: 0', ' asparagus'], axis=1, inplace=True)
store.head(2)

,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,body spray,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,True,True,False,True,False,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


**Convert boolean values in store dataframe to int and assign it back to store**


In [7]:
store = store.astype('int')
store.head(2)

,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,body spray,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,1,1,0,1,0,0,0,0,0,0,...,0,1,0,0,1,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Give row names, R0, R1 ...etc, to store dataframe**

In [8]:
rnames = ['R' +str(i) for i in range(0,store.shape[0])]
store.index = rnames
print (store.shape)
store.head(3)

(7501, 119)


,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,body spray,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
R0,1,1,0,1,0,0,0,0,0,0,...,0,1,0,0,1,0,0,1,0,0
R1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
R2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


we have total 7501 transactions, and 120 columns (or food items)

### E.3 Preliminary exploration and visualization

**Find top 5 popular items and print them as a dataframe with item, freq, and support as columns**

In [9]:
popular_items = pd.DataFrame(store.sum(0).sort_values(ascending=False)).reset_index()
popular_items.rename(columns = {'index':'item', 0:'freq'}, inplace=True)
popular_items['support'] = popular_items['freq']/store.shape[0]
popular_items.head(5)

,item,freq,support
0,mineral water,1788,0.238368
1,eggs,1348,0.179709
2,spaghetti,1306,0.174110
3,french fries,1282,0.170911
4,chocolate,1229,0.163845


## Step 2. Short-list frequently occuring items and item sets by choosing a support level

### E.4 Obtain frequently occuring items/item sets which occur atleast 0.5% of the time. Store them in 'freq_items'

**Obtain the frequent items and print the total number of item sets generated**

In [10]:
freq_items = apriori(store, min_support=0.005, use_colnames=True, max_len = None)
freq_items.shape[0]

/Volumes/SSD Workspace/tech_life/github/Data-Science-Adventures/.venv/lib/python3.7/site-packages/mlxtend/frequent_patterns/fpcommon.py:113: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  DeprecationWarning,


725

**show the first 5 rows of frequently occuring items dataframe**

In [11]:
freq_items.head(5)

,support,itemsets
0,0.020397,(almonds)
1,0.008932,(antioxydant juice)
2,0.033329,(avocado)
3,0.008666,(bacon)
4,0.010799,(barbecue sauce)


**Sort the 'freq_items' dataframe to show only top 10 most frequently occuring items**

In [12]:
freq_items.sort_values('support', ascending=False).head(10)

,support,itemsets
60,0.238368,(mineral water)
27,0.179709,(eggs)
83,0.174110,(spaghetti)
33,0.170911,(french fries)
20,0.163845,(chocolate)
44,0.132116,(green tea)
59,0.129583,(milk)
45,0.098254,(ground beef)
39,0.095321,(frozen vegetables)
68,0.095054,(pancakes)


## Step 3: From the frequently occuring item sets, generate association rules by choosing a metric

### E.5 Generate association rules which have a lift value of atleast 2

**Obtain the rules from 'association_rules' library and print the number of rules generated**

In [13]:
arules = association_rules(freq_items, metric='lift', min_threshold=2)
arules.shape[0]

482

### E.6 Sort Rules based on confidence and see top 10 rules

In [14]:
arules.sort_values('confidence', ascending=False).iloc[0:10,:-2]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage
295,"(frozen vegetables, soup)",(mineral water),0.007999,0.238368,0.005066,0.633333,2.656954,0.003159
450,"(olive oil, soup)",(mineral water),0.008932,0.238368,0.005199,0.582090,2.441976,0.003070
285,"(frozen vegetables, olive oil)",(mineral water),0.011332,0.238368,0.006532,0.576471,2.418404,0.003831
421,"(milk, soup)",(mineral water),0.015198,0.238368,0.008532,0.561404,2.355194,0.004909
186,"(chocolate, soup)",(mineral water),0.010132,0.238368,0.005599,0.552632,2.318395,0.003184
199,"(eggs, cooking oil)",(mineral water),0.011732,0.238368,0.006399,0.545455,2.288286,0.003603
263,"(frozen vegetables, ground beef)",(mineral water),0.016931,0.238368,0.009199,0.543307,2.279277,0.005163
431,"(milk, turkey)",(mineral water),0.011332,0.238368,0.006133,0.541176,2.270338,0.003431
469,"(spaghetti, soup)",(mineral water),0.014265,0.238368,0.007466,0.523364,2.195614,0.004065
396,"(ground beef, shrimp)",(spaghetti),0.011465,0.174110,0.005999,0.523256,3.005315,0.004003


### E.7 Identiy any rare strong relationships

In [15]:
arules.sort_values('lift', ascending=False).iloc[0:10,:-2]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage
22,(pasta),(escalope),0.015731,0.079323,0.005866,0.372881,4.700812,0.004618
23,(escalope),(pasta),0.079323,0.015731,0.005866,0.073950,4.700812,0.004618
69,(shrimp),(pasta),0.071457,0.015731,0.005066,0.070896,4.506672,0.003942
68,(pasta),(shrimp),0.015731,0.071457,0.005066,0.322034,4.506672,0.003942
67,(whole wheat pasta),(olive oil),0.029463,0.065858,0.007999,0.271493,4.122410,0.006059
66,(olive oil),(whole wheat pasta),0.065858,0.029463,0.007999,0.121457,4.122410,0.006059
351,(ground beef),"(herb & pepper, spaghetti)",0.098254,0.016264,0.006399,0.065129,4.004360,0.004801
346,"(herb & pepper, spaghetti)",(ground beef),0.016264,0.098254,0.006399,0.393443,4.004360,0.004801
345,(ground beef),"(mineral water, herb & pepper)",0.098254,0.017064,0.006666,0.067843,3.975683,0.004989
342,"(mineral water, herb & pepper)",(ground beef),0.017064,0.098254,0.006666,0.390625,3.975683,0.004989
